In [0]:
dbutils.widgets.dropdown("fail_on_error", "yes", ["yes", "no"])
dbutils.widgets.text("batch_id", "")
dbutils.widgets.dropdown("run_idempotency_test", "no", ["no", "yes"])

In [0]:
# ---- settings and small helpers (the same block is in every notebook) ----
import json, os, re, uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

CATALOG = "workspace"      # change these two lines if your catalog / schema have other names
SCHEMA = "default"

def T(name):
    """Full table name, e.g. T("silver_quintile_index") -> workspace.default.silver_quintile_index"""
    return f"{CATALOG}.{SCHEMA}.{name}"

AUDIT_TABLE = T("pipeline_execution_logs")
QUARANTINE_TABLE = T("quarantine_records")

def now():
    """Current time (UTC)."""
    return datetime.now(timezone.utc).replace(tzinfo=None)

def sql_text(text):
    """Makes a value safe inside a SQL string: O'Brien -> O''Brien"""
    return str(text).replace("'", "''")

def norm_date(text):
    """Turns 1.10.2026, 01-10-2026 or 2026-10-01 into 01.10.2026"""
    for pattern in ["%d.%m.%Y", "%d-%m-%Y", "%Y-%m-%d"]:
        try:
            return datetime.strptime(str(text).strip(), pattern).strftime("%d.%m.%Y")
        except ValueError:
            pass
    raise ValueError(f"Cannot read the date '{text}'. Use dd.mm.yyyy, for example 01.10.2026")

# ---- execution log: one row per layer, per file/week, per run ----
AUDIT_COLUMNS = ("log_id string, batch_id string, load_type string, layer string, file_processed string, "
                 "target_table string, start_time timestamp, end_time timestamp, status string, "
                 "rows_read long, rows_inserted long, rows_updated long, rows_quarantined long, "
                 "error_message string, message string")

def log_audit(batch_id, load_type, layer, file_processed, target_table, start_time, status,
              rows_read=0, rows_inserted=0, rows_updated=0, rows_quarantined=0, message=""):
    """status is Success, Warning (success with notes), Skipped or Failure."""
    message = (message or "")[:1500]
    error_message = message if status == "Failure" else None      # error_message is only filled for failures
    row = (str(uuid.uuid4()), batch_id, load_type, layer, str(file_processed), target_table,
           start_time, now(), status, int(rows_read), int(rows_inserted), int(rows_updated),
           int(rows_quarantined), error_message, message)
    spark.createDataFrame([row], AUDIT_COLUMNS).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

# ---- quarantine: rows (or whole files) that failed a check, saved together with the reason ----
def quarantine_from_list(items):
    """items = list of (record_ref, reason, record_json)"""
    return spark.createDataFrame(items, "record_ref string, reason string, record_json string")

def save_quarantine(df, layer, source_key, batch_id):
    """Replaces the earlier quarantine rows of the same layer and file, so re-runs never pile up duplicates.
    df needs the columns record_ref, reason and record_json."""
    spark.sql(f"DELETE FROM {QUARANTINE_TABLE} WHERE layer = '{layer}' AND source_file = '{sql_text(source_key)}'")
    rows = df.select(F.expr("uuid()").alias("quarantine_id"), F.lit(batch_id).alias("batch_id"),
                     F.lit(layer).alias("layer"), F.lit(source_key).alias("source_file"),
                     "record_ref", "reason", "record_json", F.current_timestamp().alias("quarantined_at"))
    rows.write.format("delta").mode("append").saveAsTable(QUARANTINE_TABLE)


In [0]:
results = []                                   # one entry per check: (name, PASS/FAIL/INFO, detail)

def check(name, passed, detail=""):
    results.append((name, "PASS" if passed else "FAIL", str(detail)[:300]))

def info(name, detail):
    results.append((name, "INFO", str(detail)[:300]))

def count(sql):
    """Runs a query that returns one number."""
    return spark.sql(sql).collect()[0][0]

ALL_TABLES = ["pipeline_execution_logs", "quarantine_records", "bronze_wfp_raw", "bronze_pbs_report",
              "bronze_pbs_annex", "silver_wfp_price_observation", "silver_quintile_index",
              "silver_item_weight_impact", "silver_city_price_observation"]
DATA_TABLES = ALL_TABLES[2:]                   # Bronze and Silver tables

wfp_bronze = T("bronze_wfp_raw")
wfp_silver = T("silver_wfp_price_observation")
quintile = T("silver_quintile_index")
weights = T("silver_item_weight_impact")
city = T("silver_city_price_observation")

# 1) every table exists
for name in ALL_TABLES:
    check(f"table exists: {name}", spark.catalog.tableExists(T(name)))
missing = [n for n in ALL_TABLES if not spark.catalog.tableExists(T(n))]
if missing:
    raise Exception(f"These tables are missing, run Schemas first: {missing}")

# 2) every Bronze and Silver row has a load_timestamp
for name in DATA_TABLES:
    check(f"load_timestamp is never empty: {name}", count(f"SELECT count(*) FROM {T(name)} WHERE load_timestamp IS NULL") == 0)

# 3) execution log: no failure in the batch we check (default: the latest batch)
batch_id = dbutils.widgets.get("batch_id").strip()
if batch_id == "":
    latest = spark.sql(f"SELECT batch_id FROM {AUDIT_TABLE} ORDER BY start_time DESC LIMIT 1").collect()
    batch_id = latest[0][0] if latest else ""
if batch_id != "":
    failures = count(f"SELECT count(*) FROM {AUDIT_TABLE} WHERE batch_id = '{batch_id}' AND status = 'Failure'")
    check(f"execution log: no Failure in batch {batch_id}", failures == 0, f"{failures} failures")

# 4) Bronze WFP
check("bronze_wfp_raw has rows", count(f"SELECT count(*) FROM {wfp_bronze}") > 0)
check("bronze_wfp_raw: source_file and batch_id are filled",
      count(f"SELECT count(*) FROM {wfp_bronze} WHERE source_file IS NULL OR batch_id IS NULL") == 0)

# 5) Silver WFP
wfp_rows = count(f"SELECT count(*) FROM {wfp_silver}")
info("silver_wfp_price_observation rows", wfp_rows)
check("silver_wfp: no duplicate keys",
      count(f"""SELECT count(*) FROM (SELECT 1 FROM {wfp_silver}
                GROUP BY country, market, commodity_name, unit, price_type, price_date HAVING count(*) > 1)""") == 0)
check("silver_wfp: price above zero, date / currency / market filled",
      count(f"""SELECT count(*) FROM {wfp_silver}
                WHERE price IS NULL OR price <= 0 OR price_date IS NULL OR currency IS NULL OR market IS NULL""") == 0)

# 6) Silver quintile index
check("silver_quintile_index has rows", count(f"SELECT count(*) FROM {quintile}") > 0)
check("silver_quintile_index: 6 rows per week (Q1-Q5 + Combined)",
      count(f"SELECT count(*) FROM (SELECT 1 FROM {quintile} GROUP BY week_ending_date HAVING count(*) <> 6)") == 0)
check("silver_quintile_index: no duplicate (week, quintile)",
      count(f"SELECT count(*) FROM (SELECT 1 FROM {quintile} GROUP BY week_ending_date, quintile HAVING count(*) > 1)") == 0)
check("silver_quintile_index: week-on-week % matches the index values",
      count(f"SELECT count(*) FROM {quintile} WHERE abs((index_value / index_prev_week - 1) * 100 - pct_change_wow) > 0.05") == 0)
check("silver_quintile_index: year-on-year % matches the index values",
      count(f"SELECT count(*) FROM {quintile} WHERE abs((index_value / index_prev_year - 1) * 100 - pct_change_yoy) > 0.05") == 0)

# 7) Silver item weights
check("silver_item_weight_impact has rows", count(f"SELECT count(*) FROM {weights}") > 0)
check("silver_item_weight_impact: no duplicate (week, item)",
      count(f"SELECT count(*) FROM (SELECT 1 FROM {weights} GROUP BY week_ending_date, item_name HAVING count(*) > 1)") == 0)
check("silver_item_weight_impact: weights add up to about 100 per week",
      count(f"""SELECT count(*) FROM (SELECT 1 FROM {weights} GROUP BY week_ending_date
                HAVING abs(sum(weight_lowest) - 100) > 0.5 OR abs(sum(weight_combined) - 100) > 0.5)""") == 0)

# 8) Silver city prices
check("silver_city_price_observation has rows", count(f"SELECT count(*) FROM {city}") > 0)
check("silver_city: no duplicate (week, item, city)",
      count(f"SELECT count(*) FROM (SELECT 1 FROM {city} GROUP BY week_ending_date, item_sl, city_code HAVING count(*) > 1)") == 0)
check("silver_city: min <= avg <= max",
      count(f"""SELECT count(*) FROM {city} WHERE (min_price IS NOT NULL AND min_price > avg_price + 0.005)
                OR (max_price IS NOT NULL AND max_price < avg_price - 0.005)""") == 0)
number_of_cities = count(f"SELECT count(DISTINCT city) FROM {city}")
check("silver_city: the city column holds city names (at most 40 different values)", number_of_cities <= 40, number_of_cities)

# 9) every week in Bronze also reached Silver
bronze_weeks = set(r[0] for r in spark.sql(f"SELECT DISTINCT week_ending_date FROM {T('bronze_pbs_report')}").collect())
silver_weeks = set(r[0] for r in spark.sql(f"SELECT DISTINCT date_format(week_ending_date, 'dd.MM.yyyy') FROM {quintile}").collect())
check("every Bronze PBS week exists in Silver", bronze_weeks <= silver_weeks, f"missing in Silver: {sorted(bronze_weeks - silver_weeks)}")

# 10) how much went to quarantine
for row in spark.sql(f"SELECT layer, count(*) AS n FROM {T('quarantine_records')} GROUP BY layer ORDER BY layer").collect():
    info(f"quarantined rows: {row['layer']}", row["n"])

# 11) optional: prove that running the same input again changes nothing
def table_counts():
    names = [T("bronze_pbs_report"), T("bronze_pbs_annex"), quintile, weights, city, wfp_silver]
    counts = [count(f"SELECT count(*) FROM {n}") for n in names]
    counts.append(count(f"SELECT count(*) FROM {wfp_silver} WHERE is_revised_flag"))
    return counts

if dbutils.widgets.get("run_idempotency_test") == "yes":
    latest_week = spark.sql(f"SELECT date_format(max(week_ending_date), 'dd.MM.yyyy') FROM {quintile}").collect()[0][0]
    before = table_counts()
    args = {"week_ending_dates": latest_week, "load_type": "full_backfill", "batch_id": "idempotency_test"}
    dbutils.notebook.run("bronzepbs", 1200, args)
    dbutils.notebook.run("silverpbs", 1200, args)
    if wfp_rows > 0:
        dbutils.notebook.run("silverwfp", 1800, {"load_type": "full_backfill", "batch_id": "idempotency_test"})
    after = table_counts()
    check(f"idempotency: running week {latest_week} and the WFP merge again changes nothing", before == after,
          f"before={before} after={after}")

# Show the result
display(spark.createDataFrame(results, "check string, result string, detail string"))
failed = [r[0] for r in results if r[1] == "FAIL"]
print(f"{sum(1 for r in results if r[1] == 'PASS')} passed, {len(failed)} failed")
if failed and dbutils.widgets.get("fail_on_error") == "yes":
    raise AssertionError(f"{len(failed)} check(s) failed: " + "; ".join(failed))
dbutils.notebook.exit(json.dumps({"status": "Failure" if failed else "Success", "failed": len(failed)}))